
# SU2 bosonic matrix model: Lambda=2, EvolvedOperatorAnsatz 


Notebook created by HLD for the work arXiv: 2503.13368 [quant-ph, hep-th]

In this notebook, we run the VQE experiments for bosonic SU(2) matrix model using EvolvedOperatorAnsatz circuits at 4 couplings with the fixed Estimator seed = 225. 

- Hamiltonian-variational-ansatz

 https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.hamiltonian_variational_ansatz

- Evolved Operator ansatz

  https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.evolved_operator_ansatz

In [6]:
import numpy as np
import pylab
import time
import matplotlib.pyplot as plt

import qiskit
from qiskit.quantum_info import Pauli
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms import NumPyEigensolver

import sys
sys.path.append('../../utility')
from vqe_run import *
from qc_ansatze import *

In [2]:
seed = 225
iterations = 300
algorithm_globals.random_seed = seed

#estimator
noiseless_estimator = AerEstimator(
    run_options={"seed": seed, "shots": 1024},
    transpile_options={"seed_transpiler": seed},
)
#storing values
counts = []
values = []
def store_intermediate_result(eval_count, parameters, mean, std):
    counts.append(eval_count)
    values.append(mean)
    
def run_qve_w_specified_optimizer(optimizer, ansatz):
    opt = optimizer(maxiter = iterations)
    vqe = VQE(noiseless_estimator, ansatz, optimizer=opt, callback=store_intermediate_result)
    result = vqe.compute_minimum_eigenvalue(operator=H4q).eigenvalue.real
    print(f"VQE result: {result:.5f}")
    return result

# QC

In [3]:
from qiskit.circuit.library import EvolvedOperatorAnsatz
ops_random = [Pauli("ZZIIII"), Pauli("IZIIZI"), Pauli("IXIXIX")]

#ops_id has no parameters
ops_id = [Pauli('IIIIII')]

ops_H = [Pauli('IIIIIZ'), Pauli('IIIIZI'), Pauli('IIIZII'),
        Pauli('IIZIII'),Pauli('IXXIXX'), Pauli('IZIIII'), 
        Pauli('XIXXIX'), Pauli('XXIXXI'), Pauli('ZIIIII')]

ops_Hpartial =  [Pauli('IIIIIZ'), Pauli('IIIZII'),
        Pauli('IXXIXX'), Pauli('IZIIII'), 
        Pauli('XIXXIX') ]

#ops_H_full = [Pauli('IIIIII'),Pauli('IIIIIZ'), Pauli('IIIIZI'), 
#        Pauli('IIIZII'),Pauli('IIZIII'), Pauli('IXXIXX'), Pauli('IZIIII'), 
#       Pauli('XIXXIX'), Pauli('XXIXXI'), Pauli('ZIIIII')]
#this is the same as above since the IIIIII op doesnt contribute any params

ev_op_r = EvolvedOperatorAnsatz(ops_random, reps=1, insert_barriers=True)
ev_op_r3 = EvolvedOperatorAnsatz(ops_random, reps=3, insert_barriers=True)

ev_op_H = EvolvedOperatorAnsatz(ops_H, reps=1, insert_barriers=True)
ev_op_H_2f = EvolvedOperatorAnsatz(ops_H, reps=2, insert_barriers=True)
ev_op_H_3f = EvolvedOperatorAnsatz(ops_H, reps=3, insert_barriers=True)

ev_op_Hp = EvolvedOperatorAnsatz(ops_Hpartial, reps=1, insert_barriers=True)
ev_op_Hp2 = EvolvedOperatorAnsatz(ops_Hpartial, reps=2, insert_barriers=True)
ev_op_Hp3 = EvolvedOperatorAnsatz(ops_Hpartial, reps=3, insert_barriers=True)
ev_op_Hp4 = EvolvedOperatorAnsatz(ops_Hpartial, reps=4, insert_barriers=True)

ansatz_list = [ev_op_r, ev_op_r3, ev_op_H, ev_op_H_2f, 
               ev_op_H_3f, ev_op_Hp, ev_op_Hp2, ev_op_Hp3, ev_op_Hp4]

ansatz_names = ['ev_op_r','ev_op_r3', 'ev_op_H', 'ev_op_H_2f',
                'ev_op_H_3f', 'ev_op_Hp', 'ev_op_Hp2', 'ev_op_Hp3','ev_op_Hp4']

print(f'number of params: {[ansatz_list[i].num_parameters for i in range(len(ansatz_list))]}')


number of params: [3, 9, 9, 18, 27, 5, 10, 15, 20]


# Coupling = 0.2

In [4]:
Hpauli =[('IIIIII', 6.15),
 ('IIIIIZ', -0.5),
 ('IIIIZI', -0.5),
 ('IIIZII', -0.5),
 ('IIZIII', -0.5),
 ('IXXIXX', -0.05),
 ('IZIIII', -0.5),
 ('XIXXIX', -0.05),
 ('XXIXXI', -0.05),
 ('ZIIIII', -0.5)]

H4q = SparsePauliOp.from_list(Hpauli)

# exactly diagonalize the system using numpy routines
solver = NumPyEigensolver(k=4)
exact_solution = solver.compute_eigenvalues(H4q)
print("Exact Result of qubit hamiltonian:", np.real(exact_solution.eigenvalues))
E_exact = np.round(np.real(exact_solution.eigenvalues)[0],5)
E_exact

Exact Result of qubit hamiltonian: [3.14807787 4.14674965 4.14674965 4.14674965]


3.14808

In [7]:
res_list = []
opt_list = [COBYLA, SPSA]
opt_list_name = ['COBYLA', 'SPSA']
for j in range(len(opt_list)):
    print('----------------------------------')
    print(f'optimizer is {opt_list_name[j]}')
    print('----------------------------------')
    for i in range(len(ansatz_list)):
        print(f'At step {(j,i)} with {ansatz_names[i]}')       
        counts = []
        values = []
        t0 = time.time()
        result = run_qve_w_specified_optimizer(COBYLA, ansatz_list[i])
        t1 = time.time()
        print(f'Length of this optimization {len(values)}, time taken = {np.round(t1-t0,3)} \n')
        counts_a = counts
        values_a = values 
        res_list.append(pd.DataFrame({f'{ansatz_names[i]}_{opt_list_name[j]}':values_a}))
        
df1 = pd.concat([res_list[i] for i in range(len(res_list))], axis = 1)
df1.to_csv('results/l2_l0.2_op_ev_seed_225.csv')

----------------------------------
optimizer is COBYLA
----------------------------------
At step (0, 0) with ev_op_r
VQE result: 3.15156
Length of this optimization 25, time taken = 0.408 

At step (0, 1) with ev_op_r3
VQE result: 3.15078
Length of this optimization 91, time taken = 0.762 

At step (0, 2) with ev_op_H
VQE result: 3.15000
Length of this optimization 102, time taken = 0.831 

At step (0, 3) with ev_op_H_2f
VQE result: 3.15059
Length of this optimization 185, time taken = 1.958 

At step (0, 4) with ev_op_H_3f
VQE result: 3.15469
Length of this optimization 300, time taken = 4.02 

At step (0, 5) with ev_op_Hp
VQE result: 3.15059
Length of this optimization 58, time taken = 0.454 

At step (0, 6) with ev_op_Hp2
VQE result: 3.15156
Length of this optimization 122, time taken = 1.068 

At step (0, 7) with ev_op_Hp3
VQE result: 3.15469
Length of this optimization 184, time taken = 1.935 

At step (0, 8) with ev_op_Hp4
VQE result: 3.15117
Length of this optimization 204, tim

#  Coupling = 0.5

In [8]:
Hpauli =[('IIIIII', 6.375),
 ('IIIIIZ', -0.5),
 ('IIIIZI', -0.5),
 ('IIIZII', -0.5),
 ('IIZIII', -0.5),
 ('IXXIXX', -0.125),
 ('IZIIII', -0.5),
 ('XIXXIX', -0.125),
 ('XXIXXI', -0.125),
 ('ZIIIII', -0.5)]

from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms import NumPyEigensolver
H4q = SparsePauliOp.from_list(Hpauli)

# exactly diagonalize the system using numpy routines
solver = NumPyEigensolver(k=4)
exact_solution = solver.compute_eigenvalues(H4q)
print("Exact Result of qubit hamiltonian:", np.real(exact_solution.eigenvalues))
E_exact = np.round(np.real(exact_solution.eigenvalues)[0],5)
E_exact

Exact Result of qubit hamiltonian: [3.36254139 4.35352431 4.35352431 4.35352431]


3.36254

In [9]:
res_list = []
opt_list = [COBYLA, SPSA]
opt_list_name = ['COBYLA', 'SPSA']
for j in range(len(opt_list)):
    print('----------------------------------')
    print(f'optimizer is {opt_list_name[j]}')
    print('----------------------------------')
    for i in range(len(ansatz_list)):
        print(f'At step {(j,i)} with {ansatz_names[i]}')       
        counts = []
        values = []
        t0 = time.time()
        result = run_qve_w_specified_optimizer(COBYLA, ansatz_list[i])
        t1 = time.time()
        print(f'Length of this optimization {len(values)}, time taken = {np.round(t1-t0,3)} \n')
        counts_a = counts
        values_a = values 
        res_list.append(pd.DataFrame({f'{ansatz_names[i]}_{opt_list_name[j]}':values_a}))
        
df1 = pd.concat([res_list[i] for i in range(len(res_list))], axis = 1)
df1.to_csv('results/l2_l0.5_op_ev_seed_225.csv')

----------------------------------
optimizer is COBYLA
----------------------------------
At step (0, 0) with ev_op_r
VQE result: 3.38037
Length of this optimization 28, time taken = 0.156 

At step (0, 1) with ev_op_r3
VQE result: 3.37109
Length of this optimization 86, time taken = 0.574 

At step (0, 2) with ev_op_H
VQE result: 3.36572
Length of this optimization 113, time taken = 0.756 

At step (0, 3) with ev_op_H_2f
VQE result: 3.36328
Length of this optimization 195, time taken = 1.845 

At step (0, 4) with ev_op_H_3f
VQE result: 3.35449
Length of this optimization 257, time taken = 3.217 

At step (0, 5) with ev_op_Hp
VQE result: 3.37451
Length of this optimization 65, time taken = 0.387 

At step (0, 6) with ev_op_Hp2
VQE result: 3.36523
Length of this optimization 103, time taken = 0.863 

At step (0, 7) with ev_op_Hp3
VQE result: 3.36768
Length of this optimization 166, time taken = 1.708 

At step (0, 8) with ev_op_Hp4
VQE result: 3.36133
Length of this optimization 262, ti